# ENSO realtime forecast v3

GPU refit of the 20 validation-selected members, using **1979-01 through 2026-07**. Historical skill is read from the benchmark models.


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd()/'RCENSO.py').exists() and (Path.cwd()/'forecast').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from forecast.forecast_real_time_v3 import run
from IPython.display import display, Image
BENCHMARK = ROOT/'gpu_runs/2026-10-11-validation-2002-2023'
OUTPUT = ROOT/'forecast/desn_forecast_v3_2026-07'


## Historical correlation

Training: 1958–1999; member selection and skill comparison: 2002–2023. Compare the full 1–19 month curve, especially 12–19 months.


In [ ]:
report = json.loads((BENCHMARK/'results.json').read_text(encoding='utf-8'))
print(report['parity'])
display(Image(filename=str(BENCHMARK/'benchmark_skill.png')))


## Latest prediction

The training target includes July 2026. July is lead 0 and the 21 future months run from August 2026 through April 2028. Forecast noise is disabled. Re-running uses the saved checkpoint if its training and selection manifest match.


In [ ]:
summary = run(BENCHMARK, batch_size=2, output=OUTPUT)
print(summary)
display(Image(filename=str(OUTPUT/'first_month_diagnosis.png')))


In [ ]:
import xarray as xr
forecast = xr.open_dataset(OUTPUT/'realtime_forecast.nc')
forecast.forecast_mean.sel(mode='Nino34').to_pandas()


Models: `models/selected_members.pkl` (20 CPU-stored weight sets; CUDA is required for this backend). Files also include the input hash, selection settings, monthly/centered forecast CSV, NetCDF member forecasts, and historical error estimates. Zenodo upload is deferred until long-lead skill reaches the benchmark target.
